In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from utcp.utcp_client import UtcpClient

from langchain_utcp_adapters import load_utcp_tools


## Create an agent with @tool 

In [ ]:

async def run_agent_example(prompt: str = "Use the multiply tool to calculate 6 * 7"):
    # 1. Initialize UTCP and discover the provider
    client = await UtcpClient.create(
        config={
            "manual_call_templates": [
                {
                    "name": "demo",
                    "call_template_type": "http",
                    "url": "http://127.0.0.1:8000/utcp",
                    "http_method": "GET",
                }
            ]
        }
    )

    try:
        # 2. Expose UTCP execution as a LangChain tool
        @tool
        async def multiply(x: int, y: int) -> int:
            """Multiply two integers using the UTCP provider."""
            result = await client.call_tool(
                "demo.multiply",
                {"x": x, "y": y}
            )
            return result["result"]

        # 3. Create LangChain's LangGraph-backed agent
        agent = create_agent(
            model=ChatOpenAI(
                model="qwen3.8:27b",
                temperature=0,
                api_key="ollama",
                base_url="http://localhost:11434/v1",
            ),
            tools=[multiply],
        )

        # 4. Invoke the agent
        response = await agent.ainvoke({
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        })

        # Display the full execution history
        for message in response["messages"]:
            message.pretty_print()

    finally:
        await client.close()

In [ ]:
await run_agent_example("Calculate 6 * 7")

## Create agent that auto-discovers tools from the UTCP client

Previous example only knows about multiple, autodiscovery finds the divide function as well

In [ ]:
async def run_auto_discovery_example(prompt: str = "Use the multiply tool to calculate 6 * 7"):
    # 1. Initialize UTCP and discover the provider
    client = await UtcpClient.create(
        config={
            "manual_call_templates": [
                {
                    "name": "demo",
                    "call_template_type": "http",
                    "url": "http://127.0.0.1:8000/utcp",
                    "http_method": "GET",
                }
            ]
        }
    )

    try:
        # 2. Auto discover UTCP tools from client
        tools = await load_utcp_tools(client)

        # 3. Create LangChain's LangGraph-backed agent
        agent = create_agent(
            model=ChatOpenAI(
                model="qwen3.8:27b",
                temperature=0,
                api_key="ollama",
                base_url="http://localhost:11434/v1",
            ),
            tools=tools,
        )

        # 4. Invoke the agent
        response = await agent.ainvoke({
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        })

        # Display the full execution history
        for message in response["messages"]:
            message.pretty_print()

    finally:
        await client.close()

In [ ]:
await run_auto_discovery_example("Multiply 12 by 12 then divide by 10")